In [41]:
import torch
import os
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")

using device: cpu


In [42]:
#defining the nn
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNet().to(device)
print(model)

NeuralNet(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=1024, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=1024, out_features=512, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=512, out_features=256, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.2, inplace=False)
    (9): Linear(in_features=256, out_features=10, bias=True)
  )
)


In [43]:
#loss function
loss_fn = nn.CrossEntropyLoss()
print(loss_fn)

#optimiser
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(optimizer)

CrossEntropyLoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [ ]:
from sklearn.utils import shuffle

#data loading and training code would go here
batch_size = 128
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])
training_data_loader = torch.utils.data.DataLoader(
    datasets.FashionMNIST(
        root = "data",
        train = True,
        download = True,
        transform = transform
    ),
    batch_size = batch_size,
    shuffle = True
    )
print("Number of batches in training data loader:")
print(len(training_data_loader))

#validation data loader
validation_data_loader = torch.utils.data.DataLoader(
    datasets.FashionMNIST(
        root="data",
        train=False,
        download=True,
        transform=transform
    ),
    batch_size=batch_size,
    shuffle=False
    )
print("Number of batches in validation data loader:")
print(len(validation_data_loader))

#test data loader

test_data_loader = torch.utils.data.DataLoader(
    datasets.FashionMNIST(
        train=False,
        root="data",
        download=True,
        transform=transform
    ),
    batch_size=batch_size,
    shuffle=False
    )
print("Number of batches in test data loader:")
print(len(test_data_loader))

Number of batches in training data loader:
938
Number of batches in validation data loader:
157
Number of batches in test data loader:
157


In [47]:
#training loop
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    running_loss = 0.0
    correct = 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        prediction = model(X)
        loss = loss_fn(prediction, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X.size(0)
        correct += (prediction.argmax(1) == y).type(torch.float).sum().item()
        if batch % 100 == 0:
            print(f"loss: {loss.item():>7f}  [{batch * len(X):>5d}/{size:>5d}]")
    avg_loss = running_loss / size
    accuracy = correct / size
    return avg_loss, accuracy

#evaluation on test set
def evaluate(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            prediction = model(X)
            test_loss += loss_fn(prediction, y).item() * X.size(0)
            correct += (prediction.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= size
    accuracy = correct / size
    print(f"Test Error: \n Accuracy: {(100*accuracy):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return test_loss, accuracy

def on_validation_Data(model, validation_data_loader, loss_fn):
    val_loss = 0
    val_correct = 0
    size = len(validation_data_loader.dataset)
    model.eval()
    with torch.no_grad():
        for X, y in validation_data_loader:
            X, y = X.to(device), y.to(device)
            prediction = model(X)
            val_loss += loss_fn(prediction, y).item() * X.size(0)
            val_correct += (prediction.argmax(1) == y).type(torch.float).sum().item()
    val_loss /= size
    accuracy = val_correct / size
    print(f"Validation Error: \n Accuracy: {(100*accuracy):>0.1f}%, Avg loss: {val_loss:>8f} \n")
    return val_loss, accuracy

In [ ]:
#main training loop
epochs = 100
for epoch in range(epochs):
    print(f"running epoch {epoch+1}\n-------------------")
    train_loss, train_acc = train(training_data_loader, model, loss_fn, optimizer)
    print(f"Train: Loss={train_loss:.4f}, Accuracy={train_acc*100:.2f}%")
    val_loss, val_acc = on_validation_Data(model, validation_data_loader, loss_fn)
    print(f"Validation: Loss={val_loss:.4f}, Accuracy={val_acc*100:.2f}%")
    test_loss, test_acc = evaluate(test_data_loader, model, loss_fn)
    print(f"Test: Loss={test_loss:.4f}, Accuracy={test_acc*100:.2f}%\n")

running epoch 1
-------------------
loss: 0.135376  [    0/60000]
loss: 0.116725  [ 6400/60000]
loss: 0.089907  [12800/60000]
loss: 0.182510  [19200/60000]
loss: 0.185669  [25600/60000]
loss: 0.279962  [32000/60000]
loss: 0.070028  [38400/60000]
loss: 0.229154  [44800/60000]
loss: 0.101747  [51200/60000]
loss: 0.140236  [57600/60000]
Train: Loss=0.1891, Accuracy=92.97%
Validation Error: 
 Accuracy: 89.1%, Avg loss: 0.391292 

Validation: Loss=0.3913, Accuracy=89.10%
Test Error: 
 Accuracy: 89.1%, Avg loss: 0.391292 

Test: Loss=0.3913, Accuracy=89.10%

running epoch 2
-------------------
loss: 0.136355  [    0/60000]
loss: 0.115299  [ 6400/60000]
loss: 0.031455  [12800/60000]
loss: 0.222804  [19200/60000]
loss: 0.115832  [25600/60000]
loss: 0.398792  [32000/60000]
loss: 0.161868  [38400/60000]
loss: 0.146205  [44800/60000]
loss: 0.179733  [51200/60000]
loss: 0.228545  [57600/60000]
Train: Loss=0.1839, Accuracy=93.23%
Validation Error: 
 Accuracy: 89.6%, Avg loss: 0.404476 

Validation: